# EpiNet Visualization Notebook

Interactive exploration of the Epigenetic Neural Network: gate activations, epigenetic state trajectories, memory utilization, and context sensitivity.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.epigenetic_network import EpigeneticNetwork
from src.models.baseline_transformer import BaselineTransformer
from src.data.dataset_loader import build_dataloaders, make_synthetic_sentiment
from src.training.train import Trainer
from src.utils.config import get_default_config

torch.manual_seed(42)
print('EpiNet visualization ready.')

## 1. Build and Train a Small EpiNet

In [ ]:
train_loader, val_loader, vocab = build_dataloaders(
    task='sentiment', n_samples=1000, max_len=32, batch_size=32, seed=42
)

epinet = EpigeneticNetwork(
    vocab_size=len(vocab), embed_dim=64, hidden_dims=[128, 64],
    epigenetic_dim=32, num_classes=2, memory_size=64, memory_dim=64,
    memory_lambda=0.1, epigenetic_alpha=0.3, num_heads=4, max_seq_len=32,
)

cfg = get_default_config()
cfg.training.epochs = 5
trainer = Trainer(epinet, cfg)
history = trainer.fit(train_loader, val_loader, task_name='viz')
print(f'Final val accuracy: {history["val_acc"][-1]:.4f}')

## 2. Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], 'b-o', label='Train')
axes[0].plot(history['val_loss'],   'r-o', label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['train_acc'], 'b-o', label='Train')
axes[1].plot(history['val_acc'],   'r-o', label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].set_ylim(0, 1)

plt.suptitle('EpiNet Training Curves')
plt.tight_layout()
plt.savefig('../results/training_curves.png', dpi=120)
plt.show()

## 3. Visualise Epigenetic Gate Activations

In [ ]:
epinet.eval()
token_ids, labels = next(iter(val_loader))

# Forward pass with two different epigenetic states
e_high = torch.ones(len(token_ids), 32) * 2.0   # high activation state
e_low  = torch.ones(len(token_ids), 32) * -2.0  # low activation state

with torch.no_grad():
    logits_high, _, info_high = epinet(token_ids, e_t=e_high, update_memory=False)
    logits_low,  _, info_low  = epinet(token_ids, e_t=e_low,  update_memory=False)

# Extract gate values from first layer, first head
gates_high = info_high['gate_values'][0][0].mean(0).cpu().numpy()  # (head_dim,)
gates_low  = info_low['gate_values'][0][0].mean(0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 3))
axes[0].bar(range(len(gates_high)), gates_high, color='steelblue')
axes[0].set_title('Gate activations — high e_t'); axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Neuron index'); axes[0].set_ylabel('Gate value g')

axes[1].bar(range(len(gates_low)), gates_low, color='tomato')
axes[1].set_title('Gate activations — low e_t'); axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Neuron index')

plt.suptitle('Epigenetic Gating: Same Weights, Different Epigenetic State')
plt.tight_layout()
plt.savefig('../results/gate_activations.png', dpi=120)
plt.show()

print(f'Mean gate (high e_t): {gates_high.mean():.3f}')
print(f'Mean gate (low  e_t): {gates_low.mean():.3f}')

## 4. Memory Store Utilisation Over Training

In [ ]:
epinet2 = EpigeneticNetwork(
    vocab_size=len(vocab), embed_dim=64, hidden_dims=[128, 64],
    epigenetic_dim=32, num_classes=2, memory_size=64, memory_dim=64,
    max_seq_len=32,
)

util_history = []
epinet2.train()
optimizer = torch.optim.Adam(epinet2.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()
e_t = None

for batch_idx, (token_ids, labels) in enumerate(train_loader):
    if e_t is not None and e_t.size(0) != token_ids.size(0):
        e_t = None
    optimizer.zero_grad()
    logits, e_t, _ = epinet2(token_ids, e_t=e_t, update_memory=True)
    e_t = e_t.detach()
    loss = criterion(logits, labels)
    loss.backward(); optimizer.step()
    util_history.append(epinet2.memory.utilization())

plt.figure(figsize=(10, 3))
plt.plot(util_history, 'g-')
plt.xlabel('Batch'); plt.ylabel('Memory utilisation')
plt.title('Memory Store Utilisation During Training')
plt.ylim(0, 1.05)
plt.tight_layout()
plt.savefig('../results/memory_utilisation.png', dpi=120)
plt.show()

## 5. Epigenetic State Trajectory

In [ ]:
from src.controllers.epigenetic_controller import EpigeneticController

ctrl = EpigeneticController(epigenetic_dim=32, alpha=0.3)
e_t  = ctrl.reset(batch_size=1)

epinet.eval()
with torch.no_grad():
    for token_ids, labels in val_loader:
        if e_t.size(0) != token_ids.size(0):
            e_t = e_t[:token_ids.size(0)] if e_t.size(0) > token_ids.size(0) else ctrl.reset(token_ids.size(0))
        _, e_t, _ = epinet(token_ids, e_t=e_t, update_memory=False)
        ctrl.record_state(e_t)
        e_t = e_t.detach()

traj = ctrl.state_trajectory()  # (T, 32)

plt.figure(figsize=(12, 4))
plt.imshow(traj.T.numpy(), aspect='auto', cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(label='State value')
plt.xlabel('Batch step'); plt.ylabel('e_t dimension')
plt.title('Epigenetic State Trajectory During Inference')
plt.tight_layout()
plt.savefig('../results/state_trajectory.png', dpi=120)
plt.show()
print(f'State entropy (last step): {ctrl.state_entropy(e_t):.3f}')